In [28]:
# connecting our Rpc

import os
from dotenv import load_dotenv
from web3 import Web3

load_dotenv()
RPC_URL = os.environ.get("RPC_URL")


connect = Web3(Web3.HTTPProvider(RPC_URL))
print(connect.is_connected()) # Used to check if the connection is established.

True


In [7]:
# hashing the event signature

from web3 import Web3
swap_topic0 = Web3.keccak(text="Swap(address,address,int256,int256,uint160,uint128,int24)").hex()


print('0x' + swap_topic0) 
swap_topic0 = '0x'+swap_topic0

0xc42079f94a6350d7e6235f29174924f928cc2ac818eb64fed8004e115fbcca67


In [ ]:
# Using the web3.py To get Transfers in the blockchain.

from_block = 26314971
to_block = 26314976


# Query the RPC node for Transfer logs within a block window
logs = connect.eth.get_logs({
    "fromBlock": from_block,
    "toBlock": to_block,
    "topics": [swap_topic0],  # Filter ALL Swap events across any Uniswap contract
})

print(len(logs))
print(logs[0])  # Print the first log entry to see its structure

175
AttributeDict({'removed': False, 'logIndex': 12, 'transactionIndex': 2, 'transactionHash': HexBytes('0xdc8f4d18df5ce1fd0d191e7e7c061602c5a7a07fed62a121abac55a144a2be3a'), 'blockHash': HexBytes('0xe16adc0deadf74e09a4d8a17e4b0ebb0a65b8151cbb1c9f51f1c071bdf15eeee'), 'blockNumber': 26314970, 'blockTimestamp': '0x6ae6479f', 'address': '0xd661623947EF2Ead12D577b47021C167d163B0c5', 'data': HexBytes('0xfffffffffffffffffffffffffffffffffffffffffffffffffffffffcf833665e000000000000000000000000000000000000000000000000503d1fd417db50000000000000000000000000000000000000005253c88f4e82b8000e0a8bff21500000000000000000000000000000000000000000000000001f6d99daf8ab18e400000000000000000000000000000000000000000000000000000000000309d7'), 'topics': [HexBytes('0xc42079f94a6350d7e6235f29174924f928cc2ac818eb64fed8004e115fbcca67'), HexBytes('0x00000000000000000000000051c72848c68a965f66fa7a88855f9f7784502a7f'), HexBytes('0x00000000000000000000000051c72848c68a965f66fa7a88855f9f7784502a7f')]})


In [9]:
# Manual Decoding

logs0 = logs[0]

print(logs0)
# print(logs0['transactionHash'])

transactionHash = logs0['transactionHash']
data = logs0['data']
topics = logs0['topics']
print('0x' + transactionHash.hex())
print('0x' + data.hex())

topic0 = topics[0]
topic1 = topics[1]
topic2 = topics[2]


print(topic0.hex())
print(topic1.hex())
print(topic2.hex())
# print(topics)

address_from = '0x' + topic1.hex()[-40:]

address_to = '0x' + topic2.hex()[-40:]

print(address_from)
print(address_to)


print(data)
print(int.from_bytes(data,byteorder='big'))
data = int.from_bytes(data,byteorder='big')
print(data)

AttributeDict({'removed': False, 'logIndex': 12, 'transactionIndex': 2, 'transactionHash': HexBytes('0xdc8f4d18df5ce1fd0d191e7e7c061602c5a7a07fed62a121abac55a144a2be3a'), 'blockHash': HexBytes('0xe16adc0deadf74e09a4d8a17e4b0ebb0a65b8151cbb1c9f51f1c071bdf15eeee'), 'blockNumber': 26314970, 'blockTimestamp': '0x6ae6479f', 'address': '0xd661623947EF2Ead12D577b47021C167d163B0c5', 'data': HexBytes('0xfffffffffffffffffffffffffffffffffffffffffffffffffffffffcf833665e000000000000000000000000000000000000000000000000503d1fd417db50000000000000000000000000000000000000005253c88f4e82b8000e0a8bff21500000000000000000000000000000000000000000000000001f6d99daf8ab18e400000000000000000000000000000000000000000000000000000000000309d7'), 'topics': [HexBytes('0xc42079f94a6350d7e6235f29174924f928cc2ac818eb64fed8004e115fbcca67'), HexBytes('0x00000000000000000000000051c72848c68a965f66fa7a88855f9f7784502a7f'), HexBytes('0x00000000000000000000000051c72848c68a965f66fa7a88855f9f7784502a7f')]})
0xdc8f4d18df5ce1fd0d191e7

In [19]:
## Decoding with ABI 

contract_address = '0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640' # USDC/WETH Uniswap V3 Pool

abi = [ {
        "anonymous": False,
        "inputs": [
            {"indexed": True, "internalType": "address", "name": "sender", "type": "address"},
            {"indexed": True, "internalType": "address", "name": "recipient", "type": "address"},
            {"indexed": False, "internalType": "int256", "name": "amount0", "type": "int256"},
            {"indexed": False, "internalType": "int256", "name": "amount1", "type": "int256"},
            {"indexed": False, "internalType": "uint160", "name": "sqrtPriceX96", "type": "uint160"},
            {"indexed": False, "internalType": "uint128", "name": "liquidity", "type": "uint128"},
            {"indexed": False, "internalType": "int24", "name": "tick", "type": "int24"},
        ],
        "name": "Swap",
        "type": "event",
    },]

contract = connect.eth.contract(address=contract_address,abi=abi)
events = contract.events.Swap()
decoded = events.process_log(logs[0])
print(decoded)

sender = decoded.args['sender']   
recipient = decoded.args['recipient']
amount0 = decoded.args['amount0']
amount1 = decoded.args['amount1']
sqrtPriceX96 = decoded.args['sqrtPriceX96']
liquidity = decoded.args['liquidity']
tick = decoded.args['tick']


AttributeDict({'args': AttributeDict({'sender': '0x51C72848c68a965f66FA7a88855F9f7784502a7F', 'recipient': '0x51C72848c68a965f66FA7a88855F9f7784502a7F', 'amount0': -13015751074, 'amount1': 5781812492407361536, 'sqrtPriceX96': 1669795595237233309695811700859216, 'liquidity': 2264635353391110372, 'tick': 199127}), 'event': 'Swap', 'logIndex': 12, 'transactionIndex': 2, 'transactionHash': HexBytes('0xdc8f4d18df5ce1fd0d191e7e7c061602c5a7a07fed62a121abac55a144a2be3a'), 'address': '0xd661623947EF2Ead12D577b47021C167d163B0c5', 'blockHash': HexBytes('0xe16adc0deadf74e09a4d8a17e4b0ebb0a65b8151cbb1c9f51f1c071bdf15eeee'), 'blockNumber': 26314970})


In [ ]:
sig = 'Swap(address,address,int256,int256,uint160,uint128,int24)'


topic0 = Web3.keccak(text=sig).hex()


from_block = 26314971
to_block = 26314976

# Query the RPC node for Swap logs within a block window
logs = connect.eth.get_logs({
    "fromBlock": from_block,
    "toBlock": to_block,
    "topics": ['0x'+topic0],  # Filter ALL Transfer events across any ERC-20 contract
})

print(len(logs))

pool_address = '0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640'
abi = [ {
        "anonymous": False,
        "inputs": [
            {"indexed": True, "internalType": "address", "name": "sender", "type": "address"},
            {"indexed": True, "internalType": "address", "name": "recipient", "type": "address"},
            {"indexed": False, "internalType": "int256", "name": "amount0", "type": "int256"},
            {"indexed": False, "internalType": "int256", "name": "amount1", "type": "int256"},
            {"indexed": False, "internalType": "uint160", "name": "sqrtPriceX96", "type": "uint160"},
            {"indexed": False, "internalType": "uint128", "name": "liquidity", "type": "uint128"},
            {"indexed": False, "internalType": "int24", "name": "tick", "type": "int24"},
        ],
        "name": "Swap",
        "type": "event"
  },]

print(logs)
contract = connect.eth.contract(address=pool_address,abi=abi)
print(contract.address)
events = contract.events.Swap()
print(events)

decoded = events.process_log(logs[0])

print(decoded)

175
[AttributeDict({'removed': False, 'logIndex': 12, 'transactionIndex': 2, 'transactionHash': HexBytes('0xdc8f4d18df5ce1fd0d191e7e7c061602c5a7a07fed62a121abac55a144a2be3a'), 'blockHash': HexBytes('0xe16adc0deadf74e09a4d8a17e4b0ebb0a65b8151cbb1c9f51f1c071bdf15eeee'), 'blockNumber': 26314970, 'blockTimestamp': '0x6ae6479f', 'address': '0xd661623947EF2Ead12D577b47021C167d163B0c5', 'data': HexBytes('0xfffffffffffffffffffffffffffffffffffffffffffffffffffffffcf833665e000000000000000000000000000000000000000000000000503d1fd417db50000000000000000000000000000000000000005253c88f4e82b8000e0a8bff21500000000000000000000000000000000000000000000000001f6d99daf8ab18e400000000000000000000000000000000000000000000000000000000000309d7'), 'topics': [HexBytes('0xc42079f94a6350d7e6235f29174924f928cc2ac818eb64fed8004e115fbcca67'), HexBytes('0x00000000000000000000000051c72848c68a965f66fa7a88855f9f7784502a7f'), HexBytes('0x00000000000000000000000051c72848c68a965f66fa7a88855f9f7784502a7f')]}), AttributeDict({'rem